# PROSTATE MACHINE LEARNING <br>
step:
*   Import dataset
*   label encoding
*   Data splitting 80% 20%
*   Data normalization min max
*   FS2 (LR-Lasso and Random Forest)
*   5 Biomarker model training (5-CV) : (LR-LR, RF-Random Forest, RF-XGBoost, RF-Neural Network)
*   External validation with 5% (LR, RF, XGBoost, NN)
*   Pathway Analysis







# Import dataset

In [21]:
import pandas as pd

rna = pd.read_csv("596_RNAmatrix.tsv", sep="\t")
sheet = pd.read_csv("samplesheet.tsv", sep="\t")


In [22]:
rna

,PPP5C,PSMC4,NSUN2,SEC61A2,RRAGB,STRN4,HOMER2,AKT2,PRUNE2,TENT4A,...,ZXDA,ZXDB,CHIC1,JPX,FTX,ZNF585B,ZNF260,ZNF225,ZNF234,PTOV1-AS1
0,10.217433,10.892011,10.764008,8.086177,8.284940,11.049385,12.521871,11.179877,12.028402,8.818706,...,6.827586,8.693187,8.392965,8.842550,10.025969,7.896644,9.159700,6.519995,6.751049,7.855551
1,10.377779,11.351635,10.217126,8.173544,9.204712,11.018508,11.941439,11.479496,11.097688,8.447079,...,6.254232,8.065317,7.588616,8.756382,9.407880,7.975135,8.562148,6.400688,6.827370,8.676903
2,10.696659,11.302865,10.406636,7.709796,8.531338,11.317314,12.275989,11.456222,12.140479,8.431624,...,6.667442,8.240317,8.233366,8.567713,9.706918,8.476467,8.857141,7.078470,7.226035,8.388472
3,10.211635,11.273275,10.396608,8.242036,8.725181,11.082314,11.960114,11.820012,10.821414,8.896989,...,6.572094,8.045937,7.542778,8.151569,9.239979,7.371037,8.349528,6.291772,6.751314,8.289089
4,10.322802,10.769079,10.289901,8.170423,8.632439,11.030789,12.325524,11.519839,10.970186,8.898647,...,6.790196,8.351455,8.390939,9.004244,10.737848,8.530063,8.960634,7.481751,7.973136,8.466805
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
591,9.682272,10.289631,10.312682,8.991794,9.920990,8.830769,13.325137,10.997602,12.890850,10.741442,...,9.352230,10.881638,9.699864,11.271056,12.924908,9.880423,9.768165,7.967604,8.325996,8.685253
592,10.084161,10.947362,10.621806,7.816751,8.442580,11.049746,12.133837,11.438902,11.122133,8.506886,...,7.106947,8.853999,8.224316,8.506886,9.590503,8.347443,9.890513,7.024853,7.159181,7.658078
593,10.308241,10.653146,10.773054,8.385387,8.258157,11.389410,11.844033,11.484630,11.262147,9.091902,...,6.820468,8.405001,8.195710,9.063391,10.426483,8.206928,9.277104,6.914908,7.677208,8.405001
594,10.712559,11.028111,10.580842,8.328869,8.905668,11.458211,10.862751,11.884749,10.654921,9.215811,...,7.335277,8.687947,8.141529,8.826479,9.846003,8.547781,8.809485,7.033348,7.443499,8.773290


In [23]:
sheet

,Sample_ID,class
0,TCGA-KK-A6E6-nonMT,Non-metastasis
1,TCGA-KK-A6E3-nonMT,Non-metastasis
2,TCGA-G9-7519-nonMT,Non-metastasis
3,TCGA-HC-A6AS-nonMT,Non-metastasis
4,TCGA-HC-8260-nonMT,Non-metastasis
...,...,...
591,DTB-121-MT,Metastasis
592,TCGA-EJ-5499-nonMT,Non-metastasis
593,TCGA-YL-A8SF-nonMT,Non-metastasis
594,TCGA-HC-7080-nonMT,Non-metastasis


# Data Labeling

In [24]:
sheet["label"] = sheet["class"].map({"Non-metastasis": 0, "Metastasis": 1})
sheet


,Sample_ID,class,label
0,TCGA-KK-A6E6-nonMT,Non-metastasis,0
1,TCGA-KK-A6E3-nonMT,Non-metastasis,0
2,TCGA-G9-7519-nonMT,Non-metastasis,0
3,TCGA-HC-A6AS-nonMT,Non-metastasis,0
4,TCGA-HC-8260-nonMT,Non-metastasis,0
...,...,...,...
591,DTB-121-MT,Metastasis,1
592,TCGA-EJ-5499-nonMT,Non-metastasis,0
593,TCGA-YL-A8SF-nonMT,Non-metastasis,0
594,TCGA-HC-7080-nonMT,Non-metastasis,0


# Data splitting

In [25]:
from sklearn.model_selection import train_test_split

X = rna  # Features (gene expression matrix)
y = sheet["label"]  # Labels

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


# Data normalization min max

In [26]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Features selection 2

In [27]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def cross_validate_model(X_train_scaled, y_train, model_type="lr", folds=5):
    """Performs k-fold cross-validation and collects classification metrics + feature importance."""

    if model_type == "lr":
        model = LogisticRegression(penalty="l1", solver="liblinear", max_iter=1000, random_state=42)

    elif model_type == "rf":
        model = RandomForestClassifier(random_state=42)
    else:
        raise ValueError("model_type must be 'lr' (Logistic Regression) or 'rf' (Random Forest)")

    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    metrics = {"recall": [], "precision": [], "f1-score": []}
    feature_importances = []
    idx = 1
    for train_idx, val_idx in skf.split(X_train_scaled, y_train):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        print(f"Fold {idx}/{folds}")
        idx += 1
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        report = classification_report(y_val, y_pred, output_dict=True)

        # Store classification metrics
        for metric in metrics.keys():
            class0 = report["0"][metrshap.plots.beeswarm(shap_values)ic]
            class1 = report["1"][metric]
            avg = report["weighted avg"][metric]
            metrics[metric].append([class0, class1, avg])

        # Store feature importance
        if model_type == "lr":
            importance = np.abs(model.coef_).flatten()  # Absolute coefficients
        else:  # Random Forest
            importance = np.abs(model.feature_importances_)  # Absolute feature importance

        feature_importances.append(importance)

    return metrics, np.array(feature_importances)

def display_performance_matrix(metrics):
    """Displays Recall, Precision, and F1-score performance matrices."""
    for metric_name, values in metrics.items():
        df = pd.DataFrame(values, columns=["Class0", "Class1", "Avg"]).T
        df.columns = [f"CV{i+1}" for i in range(df.shape[1])]
        df["Avg"] = df.mean(axis=1)
        print(f"\n{metric_name.capitalize()}")
        print(df.to_string())

def harvest_feature_importance(X_train, feature_importances):
    """Formats feature importance into a DataFrame with correct feature names."""
    feature_df = pd.DataFrame(
        feature_importances.T,
        columns=[f"CV{i+1}" for i in range(feature_importances.shape[0])],
        index=X_train.columns  # Fix: Use original feature names
    )
    feature_df["Avg"] = feature_df.mean(axis=1)  # Compute average importance across folds
    return feature_df


# Example usage:
# metrics, feature_importances = cross_validate_model(X_train_scaled, y_train, model_type="lr")  # or "rf"
# display_performance_matrix(metrics)
# feature_df = harvest_feature_importance(X_train_scaled, feature_importances)
# print(feature_df)


In [64]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

def external_validation(model, X_test_scaled, y_test):
    """Evaluates the best model on an external test set and displays classification metrics (class 1 and weighted)."""

    # Predict on test data
    y_pred = model.predict(X_test_scaled)

    # Generate classification report
    report = classification_report(y_test, y_pred, output_dict=True)
    cm = confusion_matrix(y_test, y_pred)
    TN, FP, FN, TP = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)  # safe fallback

    # Compute accuracy and specificity
    accuracy = accuracy_score(y_test, y_pred)
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

    # Extract class 1 and weighted average metrics
    metrics = {
        "Class 1": {
            "Accuracy": accuracy,
            "Precision": report["1"]["precision"],
            "Recall": report["1"]["recall"],
            "F1-score": report["1"]["f1-score"],
            "Specificity": specificity
        },
        "Weighted Avg": {
            "Accuracy": accuracy,
            "Precision": report["weighted avg"]["precision"],
            "Recall": report["weighted avg"]["recall"],
            "F1-score": report["weighted avg"]["f1-score"],
            "Specificity": "N/A"
        }
    }

    metrics_df = pd.DataFrame(metrics).T
    print(metrics_df)

    return metrics_df


## 1) LR-Lasso  

In [66]:
metrics1, feature_importances1 = cross_validate_model(X_train_scaled, y_train, "lr")  # or "rf"
display_performance_matrix(metrics1)


Evaluating model: Model


KeyError: "None of [Index([ 0,  1,  2,  3,  4,  5,  7,  8,  9, 10, 11, 12, 13, 16, 17, 20, 21, 22,\n       25, 26, 27, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44,\n       46, 47, 48, 49, 50, 51, 52, 54, 56, 57, 58, 59, 60, 61, 62, 64, 66, 67,\n       68, 69],\n      dtype='int64')] are in the [columns]"

In [29]:
feature_df1 = harvest_feature_importance(X_train, feature_importances1)
feature_df1["Std"] = feature_df1.iloc[:, :-2].std(axis=1)  # Compute standard deviation correctly

# Reorder columns to move "Std" after "Avg"
feature_df1 = feature_df1[[col for col in feature_df1.columns if col not in ["Avg", "Std"]] + ["Avg", "Std"]]

# Sort by "Avg" in descending order
feature_df1 = feature_df1.sort_values(by="Avg", ascending=False)

feature_df1


,CV1,CV2,CV3,CV4,CV5,Avg,Std
MYH11,6.423377,5.903639,5.127202,5.281999,5.809642,5.709172,0.596252
PRMT1,4.259685,3.627382,4.340031,3.687639,5.077802,4.198508,0.373119
TENT4A,3.934763,3.192486,4.098295,5.008275,4.159083,4.078580,0.745866
ZNF585B,1.949914,1.901872,2.615755,2.132615,1.786952,2.077422,0.326002
STRN4,0.892695,2.270441,2.058537,2.597093,0.818214,1.727396,0.741844
FTX,2.013200,1.222929,1.834310,0.947718,1.842201,1.572072,0.502274
ZXDA,0.883543,2.860247,0.693624,1.273149,1.240188,1.390150,0.985064
AKT2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
SEC61A2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
RRAGB,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [30]:
feature_df1.to_excel("feature_importance_LR.xlsx", index=True)


## RF

In [31]:
metrics2, feature_importances2 = cross_validate_model(X_train_scaled, y_train, model_type="rf")  # or "rf"
display_performance_matrix(metrics2)

Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5

Recall
        CV1  CV2       CV3  CV4  CV5       Avg
Class0  1.0  1.0  0.987342  1.0  1.0  0.997468
Class1  1.0  1.0  1.000000  1.0  1.0  1.000000
Avg     1.0  1.0  0.989474  1.0  1.0  0.997895

Precision
        CV1  CV2       CV3  CV4  CV5       Avg
Class0  1.0  1.0  1.000000  1.0  1.0  1.000000
Class1  1.0  1.0  0.941176  1.0  1.0  0.988235
Avg     1.0  1.0  0.990093  1.0  1.0  0.998019

F1-score
        CV1  CV2       CV3  CV4  CV5       Avg
Class0  1.0  1.0  0.993631  1.0  1.0  0.998726
Class1  1.0  1.0  0.969697  1.0  1.0  0.993939
Avg     1.0  1.0  0.989600  1.0  1.0  0.997920


In [32]:
feature_df2 = harvest_feature_importance(X_train, feature_importances2)
feature_df2["Std"] = feature_df2.iloc[:, :-2].std(axis=1)  # Compute standard deviation correctly

# Reorder columns to move "Std" after "Avg"
feature_df2 = feature_df2[[col for col in feature_df2.columns if col not in ["Avg", "Std"]] + ["Avg", "Std"]]

# Sort by "Avg" in descending order
feature_df2 = feature_df2.sort_values(by="Avg", ascending=False)

feature_df2


,CV1,CV2,CV3,CV4,CV5,Avg,Std
PRMT1,0.100691,0.138882,0.102184,0.109876,0.122121,0.114751,0.017777
STRN4,0.103744,0.119073,0.106158,0.118412,0.114077,0.112293,0.008028
FTX,0.119853,0.090014,0.114516,0.086156,0.091394,0.100387,0.017014
ZXDB,0.103286,0.088770,0.101176,0.101429,0.098101,0.098552,0.006664
TENT4A,0.056987,0.082622,0.093330,0.143652,0.080771,0.091472,0.036355
ZNF585B,0.063385,0.071975,0.069511,0.034090,0.057312,0.059255,0.017477
JPX,0.066451,0.049434,0.063191,0.045861,0.068771,0.058742,0.010110
ZXDA,0.066584,0.055302,0.044212,0.043994,0.053432,0.052705,0.010759
SEC61A2,0.060687,0.041643,0.051480,0.061338,0.040169,0.051063,0.009263
MYH11,0.044137,0.046151,0.041931,0.045077,0.054811,0.046421,0.001795


# 3 biomarker model training

In [33]:
biomarker = ['MYH11','PRMT1', 'TENT4A']
# biomarker5 = ['MYH11', 'PRMT1', 'TENT4A', 'STRN4', 'FTX']
X_train_scaled_biomarker = X_train_scaled[:, [rna.columns.get_loc(col) for col in biomarker]]
X_test_scaled_biomarker = X_test_scaled[:, [rna.columns.get_loc(col) for col in biomarker]]

In [34]:
def cross_validate_model(X_train_scaled, y_train, models, folds=5):
    """Performs k-fold cross-validation and selects the best model based on F1-score."""

    # If a single model is provided, convert it to a dictionary
    if not isinstance(models, dict):
        models = {"Model": models}  # Wrap single model in a dictionary

    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)

    best_model = None
    best_f1 = -1
    model_scores = {}
    best_metrics = None

    for model_name, model in models.items():
        print(f"Evaluating model: {model_name}")
        metrics = {"recall": [], "precision": [], "f1-score": []}

        for train_idx, val_idx in skf.split(X_train_scaled, y_train):
            X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
            y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_val)
            report = classification_report(y_val, y_pred, output_dict=True)

            # Store classification metrics
            for metric in metrics.keys():
                class0 = report["0"][metric]
                class1 = report["1"][metric]
                avg = report["weighted avg"][metric]
                metrics[metric].append([class0, class1, avg])

        # Compute average F1-score across folds
        avg_f1 = np.mean([x[2] for x in metrics["f1-score"]])  # Weighted avg F1-score
        model_scores[model_name] = avg_f1

        # Update best model
        if avg_f1 > best_f1:
            best_f1 = avg_f1
            best_model = model
            best_metrics = metrics  # Save metrics for best model

    print(f"Best Model: {best_model} with F1-score: {best_f1:.4f}")
    return best_model, best_metrics  # Return detailed metrics for best model


## LR

In [35]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)

best_lr, metrics_lr = cross_validate_model(X_train_scaled_biomarker, y_train, log_reg)

display_performance_matrix(metrics_lr)  # Now it gets detailed metrics


Evaluating model: Model
Best Model: LogisticRegression(max_iter=1000, random_state=42) with F1-score: 0.9979

Recall
             CV1  CV2  CV3  CV4  CV5       Avg
Class0  0.987500  1.0  1.0  1.0  1.0  0.997500
Class1  1.000000  1.0  1.0  1.0  1.0  1.000000
Avg     0.989583  1.0  1.0  1.0  1.0  0.997917

Precision
             CV1  CV2  CV3  CV4  CV5       Avg
Class0  1.000000  1.0  1.0  1.0  1.0  1.000000
Class1  0.941176  1.0  1.0  1.0  1.0  0.988235
Avg     0.990196  1.0  1.0  1.0  1.0  0.998039

F1-score
             CV1  CV2  CV3  CV4  CV5       Avg
Class0  0.993711  1.0  1.0  1.0  1.0  0.998742
Class1  0.969697  1.0  1.0  1.0  1.0  0.993939
Avg     0.989708  1.0  1.0  1.0  1.0  0.997942


## SVM

In [36]:
# import svc
from sklearn.svm import SVC
svm_rbf = SVC(kernel="rbf", probability=True, random_state=42)
best_svm, metrics_svm = cross_validate_model(X_train_scaled_biomarker, y_train, svm_rbf)
display_performance_matrix(metrics_svm)

Evaluating model: Model
Best Model: SVC(probability=True, random_state=42) with F1-score: 1.0000

Recall
        CV1  CV2  CV3  CV4  CV5  Avg
Class0  1.0  1.0  1.0  1.0  1.0  1.0
Class1  1.0  1.0  1.0  1.0  1.0  1.0
Avg     1.0  1.0  1.0  1.0  1.0  1.0

Precision
        CV1  CV2  CV3  CV4  CV5  Avg
Class0  1.0  1.0  1.0  1.0  1.0  1.0
Class1  1.0  1.0  1.0  1.0  1.0  1.0
Avg     1.0  1.0  1.0  1.0  1.0  1.0

F1-score
        CV1  CV2  CV3  CV4  CV5  Avg
Class0  1.0  1.0  1.0  1.0  1.0  1.0
Class1  1.0  1.0  1.0  1.0  1.0  1.0
Avg     1.0  1.0  1.0  1.0  1.0  1.0


In [37]:
# import svc
from sklearn.svm import SVC
svm_rbf = SVC(kernel="rbf", probability=True, random_state=42)
best_svm, metrics_svm = cross_validate_model(X_train_scaled_biomarker, y_train, svm_rbf)
display_performance_matrix(metrics_svm)

Evaluating model: Model
Best Model: SVC(probability=True, random_state=42) with F1-score: 1.0000

Recall
        CV1  CV2  CV3  CV4  CV5  Avg
Class0  1.0  1.0  1.0  1.0  1.0  1.0
Class1  1.0  1.0  1.0  1.0  1.0  1.0
Avg     1.0  1.0  1.0  1.0  1.0  1.0

Precision
        CV1  CV2  CV3  CV4  CV5  Avg
Class0  1.0  1.0  1.0  1.0  1.0  1.0
Class1  1.0  1.0  1.0  1.0  1.0  1.0
Avg     1.0  1.0  1.0  1.0  1.0  1.0

F1-score
        CV1  CV2  CV3  CV4  CV5  Avg
Class0  1.0  1.0  1.0  1.0  1.0  1.0
Class1  1.0  1.0  1.0  1.0  1.0  1.0
Avg     1.0  1.0  1.0  1.0  1.0  1.0


# Neural Network

In [38]:
!pip install scikeras

In [39]:
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Define Neural Network Model
def build_nn_model():
    model = Sequential([
        Dense(3, activation="relu", input_shape=(3,)),
        Dense(3, activation="relu", input_shape=(5,)),
        Dense(1, activation="sigmoid")  # Output layer for binary classification
    ])

    # Compile model
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

# Wrap TensorFlow model for compatibility with cross-validation
nn_model = KerasClassifier(model=build_nn_model, epochs=50, batch_size=8, verbose=0)

# Perform 5-fold cross-validation
#metrics_nn = cross_validate_model(X_train_scaled_biomarker, y_train, nn_model)

# Display performance matrix
#display_performance_matrix(metrics_nn)


In [40]:
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Define Neural Network Model
def build_nn_model():
    model = Sequential([
        Dense(3, activation="relu", input_shape=(3,)),
        Dense(3, activation="relu", input_shape=(5,)),
        Dense(1, activation="sigmoid")  # Output layer for binary classification
    ])

    # Compile model
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

# Wrap TensorFlow model for compatibility with cross-validation
nn_model = KerasClassifier(model=build_nn_model, epochs=50, batch_size=8, verbose=0)
best_nn, metrics_nn = cross_validate_model(X_train_scaled_biomarker, y_train, nn_model)
display_performance_matrix(metrics_nn)

Evaluating model: Model


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/lo

Best Model: KerasClassifier(
	model=<function build_nn_model at 0x78f8b0f5ad40>
	build_fn=None
	warm_start=False
	random_state=None
	optimizer=rmsprop
	loss=None
	metrics=None
	batch_size=8
	validation_batch_size=None
	verbose=0
	callbacks=None
	validation_split=0.0
	shuffle=True
	run_eagerly=False
	epochs=50
	class_weight=None
) with F1-score: 0.9469

Recall
             CV1  CV2       CV3       CV4  CV5       Avg
Class0  0.987500  1.0  1.000000  0.987342  1.0  0.994968
Class1  1.000000  1.0  0.000000  1.000000  1.0  0.800000
Avg     0.989583  1.0  0.831579  0.989474  1.0  0.962127

Precision
             CV1  CV2       CV3       CV4  CV5       Avg
Class0  1.000000  1.0  0.831579  1.000000  1.0  0.966316
Class1  0.941176  1.0  0.000000  0.941176  1.0  0.776471
Avg     0.990196  1.0  0.691524  0.990093  1.0  0.934363

F1-score
             CV1  CV2       CV3       CV4  CV5       Avg
Class0  0.993711  1.0  0.908046  0.993631  1.0  0.979077
Class1  0.969697  1.0  0.000000  0.969697  1.0 

# External Validation

In [41]:
X_test_scaled_biomarker = X_test_scaled[:, [rna.columns.get_loc(col) for col in biomarker]]

In [42]:
from sklearn.metrics import classification_report

def external_validation(model, X_test_scaled, y_test):
    """Evaluates the best model on an external test set and displays classification metrics."""

    # Predict on test data
    y_pred = model.predict(X_test_scaled)

    # Generate classification report
    report = classification_report(y_test, y_pred, output_dict=True)

    # Extract metrics for each class and weighted average
    metrics = {
        "Class 0": {
            "Precision": report["0"]["precision"],
            "Recall": report["0"]["recall"],
            "F1-score": report["0"]["f1-score"]
        },
        "Class 1": {
            "Precision": report["1"]["precision"],
            "Recall": report["1"]["recall"],
            "F1-score": report["1"]["f1-score"]
        },
        "Weighted Avg": {
            "Precision": report["weighted avg"]["precision"],
            "Recall": report["weighted avg"]["recall"],
            "F1-score": report["weighted avg"]["f1-score"]
        }
    }

    # Convert to DataFrame for better visualization
    metrics_df = pd.DataFrame(metrics).T
    print(metrics_df)

    return metrics_df  # Return the metrics for further analysis if needed


In [43]:
# Perform external validation
metrics_test_lr = external_validation(best_lr, X_test_scaled_biomarker, y_test)

              Precision    Recall  F1-score
Class 0        0.990099  1.000000  0.995025
Class 1        1.000000  0.950000  0.974359
Weighted Avg   0.991749  0.991667  0.991581


In [44]:
# Perform external validation
metrics_test_svm = external_validation(best_svm, X_test_scaled_biomarker, y_test)

              Precision    Recall  F1-score
Class 0        0.990000  0.990000  0.990000
Class 1        0.950000  0.950000  0.950000
Weighted Avg   0.983333  0.983333  0.983333


In [45]:
# Perform external validation
metrics_test_nn = external_validation(best_nn, X_test_scaled_biomarker, y_test)

              Precision    Recall  F1-score
Class 0        0.990099  1.000000  0.995025
Class 1        1.000000  0.950000  0.974359
Weighted Avg   0.991749  0.991667  0.991581


# Export Model

In [46]:
import joblib

# Save the best model
joblib.dump(best_lr, "best_lr_model.pkl")
print("best_lr saved as best_lr_model.pkl")

joblib.dump(best_svm, "best_svm_model.pkl")
print("best_svm saved as best_svm_model.pk")

joblib.dump(best_nn, "best_nn_model.pkl")
print("best_nn saved as best_nn_model.pk")


best_lr saved as best_lr_model.pkl
best_svm saved as best_svm_model.pk
best_nn saved as best_nn_model.pk


# Save workspace

In [47]:
!pip install dill


In [48]:
#save
import dill
filename = "workspace.pkl"

# Save all variables
with open(filename, "wb") as f:
    dill.dump_session(f)

print("Workspace saved as workspace.pkl")


Workspace saved as workspace.pkl


In [49]:

import dill

# Load saved workspace
with open("workspace.pkl", "rb") as f:
    dill.load_session(f)

print("Workspace restored!")


Workspace restored!


# SHAP

In [57]:
import shap
import numpy as np
import matplotlib.pyplot as plt

def shap_explanation_plots_nn(model, X_test_scaled_biomarker, y_test, gene_names, background_size=100):
    """
    SHAP beeswarm + waterfall for neural networks using DeepExplainer.
    """

    # Convert y_test to pandas Series
    if isinstance(y_test, np.ndarray):
        y_test_series = pd.Series(y_test, index=X_test_scaled_biomarker.index)
    else:
        y_test_series = y_test

    # Ensure clean test data
    mask = X_test_scaled_biomarker.notna().all(axis=1) & y_test_series.notna()
    X_test_clean = X_test_scaled_biomarker[mask]
    y_test_clean = y_test_series[mask]

    # Background: random subset
    X_background = X_test_clean.sample(n=min(background_size, len(X_test_clean)), random_state=42)

    # SHAP DeepExplainer (for Keras/TensorFlow models)
    explainer = shap.DeepExplainer(model, X_background.to_numpy())
    shap_values = explainer.shap_values(X_test_clean.to_numpy())[0]

    # Beeswarm plot
    shap.summary_plot(shap_values, X_test_clean, feature_names=gene_names, show=True)

    # Waterfall plots
    for class_label in [0, 1]:
        sample_indices = y_test_clean[y_test_clean == class_label].sample(n=3, random_state=42).index
        for i, idx in enumerate(sample_indices):
            shap_values_single = shap_values[idx]
            expected_value = explainer.expected_value[0]
            print(f"Waterfall plot for class {class_label}, sample {i+1}")
            shap.plots._waterfall.waterfall_legacy(expected_value, shap_values_single, feature_names=gene_names)
            plt.show()


In [58]:
explainer = shap.TreeExplainer(best_nn)
shap_values = explainer.shap_values(X_test_scaled_biomarker)
shap.summary_plot(shap_values[1], X_test_scaled_biomarker, feature_names=gene_names)


InvalidModelError: Model type not yet supported by TreeExplainer: <class 'scikeras.wrappers.KerasClassifier'>

In [52]:
y_test

,label
88,0
584,1
189,0
20,0
295,0
...,...
525,1
363,0
257,0
252,0


In [59]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# ===== 1. Load and preprocess Iris data =====
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

# For binary classification, select only class 0 and 1
binary_mask = y < 2
X_binary = X[binary_mask]
y_binary = y[binary_mask]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_binary, y_binary, test_size=0.3, random_state=42)

# Standardize
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

# ===== 2. Train Neural Network =====
model = MLPClassifier(hidden_layer_sizes=(10,), max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# ===== 3. SHAP with KernelExplainer =====
# Define prediction function that returns class 1 probability
def predict_fn(x):
    return model.predict_proba(x)[:, 1]

# Background sample (use 50 training examples)
X_background = X_train_scaled.sample(50, random_state=42)

# Create SHAP explainer
explainer = shap.KernelExplainer(predict_fn, X_background)

# Explain all test samples
shap_values = explainer.shap_values(X_test_scaled)[0]

# ===== 4. Beeswarm Plot =====
shap.summary_plot(shap_values, X_test_scaled, plot_type="dot", show=True)

# ===== 5. Waterfall Plots =====
y_test_series = pd.Series(y_test, index=X_test_scaled.index)
for class_label in [0, 1]:
    indices = y_test_series[y_test_series == class_label].sample(3, random_state=class_label).index
    for i, idx in enumerate(indices):
        print(f"Waterfall plot for class {class_label}, sample {i+1}")
        shap.plots._waterfall.waterfall_legacy(
            explainer.expected_value, shap_values[idx], feature_names=X.columns.tolist()
        )
        plt.show()


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


  0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/v

AssertionError: Summary plots need a matrix of shap_values, not a vector.

<Figure size 640x480 with 0 Axes>